# CelebA Gender Classification

Binary image classification on the CelebA dataset: predict whether a celebrity photo is male or female. The challenge is limited data — 2,000 training, 1,000 validation, and 1,000 test images (218×178 px).

**Approach:** (1) train a CNN from scratch with data augmentation, (2) fast feature extraction with frozen VGG16, (3) fine-tune VGG16 on the classification task.

## 1. Environment Setup

Verify TensorFlow/Keras version.


In [ ]:
import keras
keras.__version__


'3.10.0'

In [ ]:
import os, shutil


## 2. Dataset Paths

Local train/validation/test folders with `male/` and `female/` subdirectories.


In [ ]:
base_dir = './hw6'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')
train_male_dir = os.path.join(train_dir, 'male')
train_female_dir = os.path.join(train_dir, 'female')
validation_male_dir = os.path.join(validation_dir, 'male')
validation_female_dir = os.path.join(validation_dir, 'female')
test_male_dir = os.path.join(test_dir, 'male')
test_female_dir = os.path.join(test_dir, 'female')


### Sanity Check

Confirm expected image counts in each split.

In [ ]:
print('total training male images:', len(os.listdir(train_male_dir)))


total training male images: 825


In [ ]:
print('total training female images:', len(os.listdir(train_female_dir)))


total training female images: 1175


In [ ]:
print('total validation male images:', len(os.listdir(validation_male_dir)))


total validation male images: 413


In [ ]:
print('total validation female images:', len(os.listdir(validation_female_dir)))


total validation female images: 587


In [ ]:
print('total test male images:', len(os.listdir(test_male_dir)))


total test male images: 412


In [ ]:
print('total test female images:', len(os.listdir(test_female_dir)))


total test female images: 588


## 3. Data Generators

Import libraries and count samples per split.


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
train_count = len(os.listdir(train_male_dir)) + len(os.listdir(train_female_dir))
val_count   = len(os.listdir(validation_male_dir)) + len(os.listdir(validation_female_dir))
test_count  = len(os.listdir(test_male_dir)) + len(os.listdir(test_female_dir))
print(train_count, val_count, test_count)


2000 1000 1000


### 3.1 CNN from Scratch

4-block Conv2D network with dropout and heavy data augmentation.


In [ ]:
img_height, img_width = 218, 178
batch_size = 32
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
test_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)
validation_generator = test_datagen.flow_from_directory(
    validation_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)
model_scratch = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu',
                  input_shape=(img_height, img_width, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model_scratch.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=['accuracy']
)
history_scratch = model_scratch.fit(
    train_generator,
    epochs=30,
    validation_data=validation_generator
)
test_loss_s, test_acc_s = model_scratch.evaluate(test_generator)
print(f"Scratch CNN test accuracy: {test_acc_s:.3f}")


Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Epoch 1/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 387ms/step - accuracy: 0.5718 - loss: 0.6805 - val_accuracy: 0.5870 - val_loss: 0.6637
Epoch 2/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 339ms/step - accuracy: 0.5950 - loss: 0.6700 - val_accuracy: 0.6140 - val_loss: 0.6191
Epoch 3/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 317ms/step - accuracy: 0.6323 - loss: 0.6382 - val_accuracy: 0.7410 - val_loss: 0.5554
Epoch 4/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 339ms/step - accuracy: 0.6742 - loss: 0.6159 - val_accuracy: 0.7540 - val_loss: 0.5462
Epoch 5/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 317ms/step - accuracy: 0.6560 - loss: 0.6256 - val_accuracy: 0.7620 - val_loss: 0.5365
Epoch 6/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 22s 344ms/step - accuracy: 0.6881 - loss: 0.5972 - val_accuracy: 0.7580 - val_loss: 0.5125
Epoch 7/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 325ms/step - accuracy: 0.6834 - loss: 0.6101 - val_accurac

### 3.2 Fast Feature Extraction (VGG16)

Extract bottleneck features with frozen VGG16, then train a dense classifier.


In [ ]:
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(img_height, img_width, 3)
)
conv_base.summary()
datagen = ImageDataGenerator(rescale=1./255)
batch_size = 32
def extract_features(directory, sample_count):
    generator = datagen.flow_from_directory(
        directory,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        class_mode='binary',
        shuffle=False
    )
    feature_shape = conv_base.output_shape[1:]
    features = np.zeros((sample_count, *feature_shape), dtype='float32')
    labels = np.zeros(sample_count, dtype='float32')
    i = 0
    for inputs_batch, labels_batch in generator:
        features_batch = conv_base.predict(inputs_batch)
        bs = len(labels_batch)
        features[i:i+bs] = features_batch
        labels[i:i+bs] = labels_batch
        i += bs
        if i >= sample_count:
            break
    return features, labels
train_features, train_labels = extract_features(train_dir, train_count)
val_features, val_labels     = extract_features(validation_dir, val_count)
test_features, test_labels   = extract_features(test_dir, test_count)
train_features = train_features.reshape((train_count, -1))
val_features   = val_features.reshape((val_count, -1))
test_features  = test_features.reshape((test_count, -1))
feature_model = models.Sequential([
    layers.Dense(256, activation='relu', input_dim=train_features.shape[1]),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])
feature_model.compile(
    optimizer=optimizers.RMSprop(learning_rate=2e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history_feat = feature_model.fit(
    train_features, train_labels,
    epochs=30,
    batch_size=32,
    validation_data=(val_features, val_labels)
)
test_loss_f, test_acc_f = feature_model.evaluate(test_features, test_labels)
print(f"Fast feature extraction test accuracy: {test_acc_f:.3f}")


Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_15 (InputLayer)     │ (None, 218, 178, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 218, 178, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 218, 178, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 109, 89, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 109, 89, 128)   │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 109, 89, 128)   │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 54, 44, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 54, 44, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 54, 44, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 54, 44, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 27, 22, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 27, 22, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 27, 22, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 27, 22, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 13, 11, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 13, 11, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 13, 11, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 13, 11, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 6, 5, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

Found 2000 images belonging to 2 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 752ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━

### 3.3 Fine-Tuning — Phase 1

Add augmentation + dense head on frozen VGG16. Save best checkpoint by validation loss.


In [ ]:
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(img_height, img_width, 3)
)
conv_base.trainable = False
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
])
inputs = keras.Input(shape=(img_height, img_width, 3))
x = data_augmentation(inputs)
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)
model.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=2e-4),
    metrics=['accuracy']
)
train_datagen_ft = ImageDataGenerator(rescale=1./255)
val_datagen_ft   = ImageDataGenerator(rescale=1./255)
train_dataset = train_datagen_ft.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)
validation_dataset = val_datagen_ft.flow_from_directory(
    validation_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="fine_tuning.keras",
        save_best_only=True,
        monitor="val_loss"
    )
]
history_top = model.fit(
    train_dataset,
    epochs=10,
    validation_data=validation_dataset,
    callbacks=callbacks
)


Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 13s 182ms/step - accuracy: 0.6888 - loss: 0.7794 - val_accuracy: 0.8670 - val_loss: 0.2999
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 177ms/step - accuracy: 0.8312 - loss: 0.3946 - val_accuracy: 0.8890 - val_loss: 0.2709
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 176ms/step - accuracy: 0.8606 - loss: 0.3229 - val_accuracy: 0.9100 - val_loss: 0.2248
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 169ms/step - accuracy: 0.8398 - loss: 0.3311 - val_accuracy: 0.9050 - val_loss: 0.2287
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 168ms/step - accuracy: 0.8605 - loss: 0.3287 - val_accuracy: 0.8780 - val_loss: 0.2754
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 173ms/step - accuracy: 0.8917 - loss: 0.2633 - val_accuracy: 0.9150 - val_loss: 0.2069
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 166ms/step - accuracy: 0.8965 - loss: 0.2555 - val_accuracy: 0.9160 - val_loss: 0.2095
Epoch 8/10
63

### 3.3 Fine-Tuning — Phase 2

Unfreeze from `block5_conv1` onward, train 50 epochs at learning rate 1e-5.


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
conv_base.trainable = True
set_trainable = False
for layer in conv_base.layers:
    if layer.name == "block5_conv1":
        set_trainable = True
    layer.trainable = set_trainable
model.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=1e-5),
    metrics=['accuracy']
)
callbacks_ft = [
    keras.callbacks.ModelCheckpoint(
        filepath="fine_tuning.keras",
        save_best_only=True,
        monitor="val_accuracy",
        mode="max"
    )
]
history_ft = model.fit(
    train_dataset,
    epochs=50,
    validation_data=validation_dataset,
    callbacks=callbacks_ft
)
test_datagen = ImageDataGenerator(rescale=1./255)
test_dataset = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)


Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 15s 200ms/step - accuracy: 0.9045 - loss: 0.2243 - val_accuracy: 0.9290 - val_loss: 0.1789
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 194ms/step - accuracy: 0.9278 - loss: 0.1763 - val_accuracy: 0.9380 - val_loss: 0.1696
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 186ms/step - accuracy: 0.9202 - loss: 0.1674 - val_accuracy: 0.9370 - val_loss: 0.1649
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 186ms/step - accuracy: 0.9374 - loss: 0.1585 - val_accuracy: 0.9350 - val_loss: 0.1729
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 219ms/step - accuracy: 0.9416 - loss: 0.1412 - val_accuracy: 0.9430 - val_loss: 0.1631
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 185ms/step - accuracy: 0.9450 - loss: 0.1301 - val_accuracy: 0.9280 - val_loss: 0.1820
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 184ms/step - accuracy: 0.9650 - loss: 0.0915 - val_accuracy: 0.9410 - val_loss: 0.1606
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 185ms/step - accuracy: 0.9691 - loss: 0.0822 - val_accu

## 4. Test Set Evaluation

Load the best fine-tuned model and report final test accuracy.

In [ ]:
model = keras.models.load_model("fine_tuning.keras")
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")


32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 107ms/step - accuracy: 0.9441 - loss: 0.2075
Test accuracy: 0.940
